<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C1%20-%20Neural%20Networks%20and%20Deep%20Learning/Week%204/Building%20your%20Deep%20Neural%20Network%20-%20Step%20by%20Step/N_Building_your_Deep_Neural_Network_Step_by_Step_v8a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C1 - Neural Networks and Deep Learning/Week 4/Building your Deep Neural Network - Step by Step"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


Setting up Colab environment...
Cloning into '/content/dls_c'...
remote: Enumerating objects: 1220, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 1220 (delta 79), reused 25 (delta 16), pack-reused 1103 (from 1)
Receiving objects: 100% (1220/1220), 199.41 MiB | 19.13 MiB/s, done.
Resolving deltas: 100% (112/112), done.
Updating files: 100% (1010/1010), done.
Filtering content: 100% (16/16), 38.95 MiB | 8.36 MiB/s, done.
/content/dls_c/C1 - Neural Networks and Deep Learning/Week 4/Building your Deep Neural Network - Step by Step
Setup complete. You can now run the rest of the notebook.


In [ ]:
import numpy as np


# =====================================================
# 1. ACTIVATION FUNCTIONS
# =====================================================

def relu(Z):
    """
    ReLU activation: max(0, Z)

    Input:
        Z : linear output of current layer

    Returns:
        A : activated output
        Z : stored for backward pass
    """
    A = np.maximum(0, Z)
    return A, Z


def sigmoid(Z):
    """
    Sigmoid activation: 1 / (1 + e^-Z)

    Input:
        Z : linear output

    Returns:
        A : activated output
        Z : stored for backward pass
    """
    A = 1 / (1 + np.exp(-Z))
    return A, Z


def relu_backward(dA, Z):
    """
    Backward for ReLU

    dA = gradient of cost w.r.t activation output
    Returns dZ
    """
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ


def sigmoid_backward(dA, Z):
    """
    Backward for Sigmoid

    dA = gradient of cost w.r.t activation output
    Returns dZ
    """
    s = 1 / (1 + np.exp(-Z))
    dZ = dA * s * (1 - s)
    return dZ


# =====================================================
# 2. PARAMETER INITIALIZATION
# =====================================================

def initialize_parameters(layer_dims):
    """
    layer_dims = list of neurons in each layer

    Example:
        [5, 4, 3, 1]

    Means:
        input = 5
        hidden1 = 4
        hidden2 = 3
        output = 1
    """
    np.random.seed(1)
    params = {}

    L = len(layer_dims)

    for l in range(1, L):
        params["W" + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        params["b" + str(l)] = np.zeros((layer_dims[l], 1))

    return params


# =====================================================
# 3. FORWARD PROPAGATION
# =====================================================

def forward_layer(A_prev, W, b, activation):
    """
    One layer forward step

    Inputs:
        A_prev : output from previous layer
        W      : weights
        b      : bias
        activation : "relu" or "sigmoid"

    Returns:
        A      : output after activation
        cache  : values needed for backward pass
    """

    Z = np.dot(W, A_prev) + b

    if activation == "relu":
        A, act_cache = relu(Z)
    else:
        A, act_cache = sigmoid(Z)

    cache = (A_prev, W, b, act_cache)
    return A, cache


def forward_model(X, params):
    """
    Full forward pass for L-layer model

    Hidden layers use ReLU
    Last layer uses Sigmoid
    """

    caches = []
    A = X
    L = len(params) // 2

    # Hidden layers
    for l in range(1, L):
        A_prev = A
        A, cache = forward_layer(A_prev,
                                 params["W"+str(l)],
                                 params["b"+str(l)],
                                 "relu")
        caches.append(cache)

    # Output layer
    AL, cache = forward_layer(A,
                              params["W"+str(L)],
                              params["b"+str(L)],
                              "sigmoid")
    caches.append(cache)

    return AL, caches


# =====================================================
# 4. COST FUNCTION
# =====================================================

def compute_cost(AL, Y):
    """
    Binary cross-entropy cost
    """

    m = Y.shape[1]

    cost = -(1/m) * np.sum(Y*np.log(AL) + (1-Y)*np.log(1-AL))
    return np.squeeze(cost)


# =====================================================
# 5. BACKWARD PROPAGATION
# =====================================================

def backward_layer(dA, cache, activation):
    """
    One layer backward step
    """

    A_prev, W, b, Z = cache
    m = A_prev.shape[1]

    # Activation backward
    if activation == "relu":
        dZ = relu_backward(dA, Z)
    else:
        dZ = sigmoid_backward(dA, Z)

    # Linear backward
    dW = (1/m) * np.dot(dZ, A_prev.T)
    db = (1/m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = np.dot(W.T, dZ)

    return dA_prev, dW, db


def backward_model(AL, Y, caches):
    """
    Full backward pass
    """

    grads = {}
    L = len(caches)

    Y = Y.reshape(AL.shape)

    # Output layer derivative
    dAL = -(np.divide(Y, AL) - np.divide(1-Y, 1-AL))

    # Last layer (sigmoid)
    dA_prev, dW, db = backward_layer(dAL, caches[L-1], "sigmoid")
    grads["dW"+str(L)] = dW
    grads["db"+str(L)] = db

    # Hidden layers
    for l in reversed(range(L-1)):
        dA_prev, dW, db = backward_layer(dA_prev, caches[l], "relu")
        grads["dW"+str(l+1)] = dW
        grads["db"+str(l+1)] = db

    return grads


# =====================================================
# 6. UPDATE PARAMETERS
# =====================================================

def update_parameters(params, grads, learning_rate):
    """
    Gradient descent update
    """

    L = len(params) // 2

    for l in range(1, L+1):
        params["W"+str(l)] -= learning_rate * grads["dW"+str(l)]
        params["b"+str(l)] -= learning_rate * grads["db"+str(l)]

    return params


# =====================================================
# 7. TRAIN MODEL
# =====================================================

def train_model(X, Y, layer_dims, learning_rate=0.01, iterations=3000):
    """
    Full training process
    """

    params = initialize_parameters(layer_dims)

    for i in range(iterations):

        # Forward
        AL, caches = forward_model(X, params)

        # Cost
        cost = compute_cost(AL, Y)

        # Backward
        grads = backward_model(AL, Y, caches)

        # Update
        params = update_parameters(params, grads, learning_rate)

        if i % 500 == 0:
            print(f"Cost after {i}: {cost}")

    return params


# =====================================================
# 8. PREDICT
# =====================================================

def predict(X, params):
    """
    Predict 0 or 1
    """

    AL, _ = forward_model(X, params)
    predictions = (AL > 0.5).astype(int)

    return predictions